# Local Validation

This notebook is used to validate a local version of the Animl deployment of small-animal-classififer against an output JSON produced by running a cloned version of the small-animal-classififer github repository. If you would like to use a different set of images or reproduce the JSON output file for the provided images, please follow the instructions here: https://github.com/agentmorris/small-animal-classifier/.

## Setting up Docker

```bash
cd models/small-animal-classifier
docker buildx build --platform linux/amd64 -t small-animal-classifier .
docker run -p 8080:8080 small-animal-classifier
```

This starts up a FastAPI server at port `8080` on your local machine. It can be reached at `http://localhost:8080`.

To check the model loaded correctly and the server is working:
`curl http://localhost:8080/ping`

## Imports

In [21]:
from pathlib import Path
from PIL import Image
import base64
import requests
import json

## Run inference on docker

In [23]:
image_dir = Path("./images")
images_b64 = {}
for image_path in image_dir.glob("*.jpg"):
    with open(image_path, "rb") as image_bytes:
        image_bytes = base64.b64encode(image_bytes.read()).decode("utf-8")
        images_b64[image_path.name] = image_bytes

print(f"loaded and encoded {len(images_b64)} images from {image_dir.resolve()}")

loaded and encoded 6 images from /Users/jesseleung/Projects/tnc-projects/animl/animl-ml/models/small-animal-classifier/validation/images


## Run inference on Docker model

In [20]:
inference_res = {}
for image_name, image_b64 in images_b64.items():
    res = requests.post(
        "http://localhost:8080/invocations",
        json={"image": image_b64}
    )
    inference_res[image_name] = res.json()
print(inference_res)

{'mouse~2004886_sa-bswa1-2024__48ff9b62-96b6-40ba-94e7-55ace1b0737e.jpg': {'blank': 0.00246220245026052, 'setup_pickup': 0.018866272643208504, 'mouse': 0.5419602990150452, 'vole': 0.0038816945161670446, 'rodent_other': 0.0028616960626095533, 'woodrat_rat': 0.004510056227445602, 'squirrel': 0.0042418790981173515, 'kangaroo_rat_pocket_mouse': 0.008118673227727413, 'chipmunk': 0.00951121374964714, 'pocket_gopher': 0.03203538432717323, 'rabbit_hare': 0.006852823309600353, 'shrew_mole': 0.009382514283061028, 'skunk': 0.01571892946958542, 'weasel': 0.020124737173318863, 'opossum': 0.03326020762324333, 'mammal_other': 0.03494733199477196, 'spiny_lizard': 0.004280491266399622, 'whiptail': 0.005609491840004921, 'snake': 0.007440414745360613, 'alligator_lizard': 0.011125569231808186, 'skink': 0.013826297596096992, 'rattlesnake': 0.024603238329291344, 'lizard_other': 0.03799258917570114, 'amphibian': 0.009054155088961124, 'bird': 0.005916958209127188, 'insect': 0.006296191364526749, 'isopod_crust

## Compare to sample output

In [66]:
with open("sample_output.json") as f:
    sample_data = json.load(f)
sample_inference = sample_data["images"]
class_names = sample_data["classification_categories"]

for image_name, res in inference_res.items():
    sample_res = next(sample for sample in sample_inference if sample["file"] == image_name)
    if sample_res is None:
        raise Exception("Did not compare the same files")
        
    sample_classifications = sample_res["detections"][0]["classifications"]
    top_3 = sorted(res.items(), key=lambda x: x[1], reverse=True)[:3]

    for classification, confidence in top_3:
        match = next(c for c in sample_classifications if class_names[c[0]] == classification)
        if match is None:
            raise Exception("Top 3 classifications did not match")
        if abs(match[1] - confidence) >= 0.00005:
            print(match[1], confidence)
            raise Exception(f"Top 3 confidence scores did not match. Expected: {match[1]}, got: {confidence}")
        print(f"Match! Expected: {match[1]}, got: {round(confidence, 4)}, rounded from: {confidence}") 

print("Classifications and confidence scores matched sample data")

    
    

Match! Expected: 0.542, got: 0.542, rounded from: 0.5419602990150452
Match! Expected: 0.0665, got: 0.0665, rounded from: 0.06652272492647171
Match! Expected: 0.038, got: 0.038, rounded from: 0.03799258917570114
Match! Expected: 0.8005, got: 0.8005, rounded from: 0.800500214099884
Match! Expected: 0.0282, got: 0.0282, rounded from: 0.028188733384013176
Match! Expected: 0.0151, got: 0.0151, rounded from: 0.01506371982395649
Match! Expected: 0.8556, got: 0.8556, rounded from: 0.8556020855903625
Match! Expected: 0.0203, got: 0.0203, rounded from: 0.02034735679626465
Match! Expected: 0.0172, got: 0.0172, rounded from: 0.017249582335352898
Match! Expected: 0.9008, got: 0.9008, rounded from: 0.9007802605628967
Match! Expected: 0.0124, got: 0.0124, rounded from: 0.012401578016579151
Match! Expected: 0.0109, got: 0.0109, rounded from: 0.010881893336772919
Match! Expected: 0.7737, got: 0.7737, rounded from: 0.7737165093421936
Match! Expected: 0.033, got: 0.033, rounded from: 0.032972682267427444